**House Prices Prediction Model.** 

**Data Loading and Initial Validation**

In [31]:
# Data Loading and Preprocessing
import pandas as pd
import numpy as np
import os



In [32]:
# Load the dataset

from pathlib import Path

data_path = Path("c:/Users/chris/Documents/House_Price_Prediction/dataset_2.csv")
print(data_path.exists(), data_path)

df = pd.read_csv(data_path)
df.head()

True c:\Users\chris\Documents\House_Price_Prediction\dataset_2.csv


,Area_SqFt,Rooms,Build_Year,Location,Street_Type,Furnishing,Property_Type,Has_Pool,Price
0,2473.192784,4.0,1992,Jaipur,Residential Lane,Furnished,Apartment,No,568486.0
1,2353.472711,4.0,2006,Indore,Corner Plot,Unfurnished,Apartment,Yes,577214.0
2,2212.222005,3.0,2012,Jaipur,Highway Facing,Semi-Furnished,Duplex,No,581300.0
3,2823.886596,6.0,1993,Lucknow,Main Road,Unfurnished,Villa,Yes,794614.0
4,1869.648721,5.0,2012,Jaipur,Corner Plot,Semi-Furnished,Apartment,No,493086.0


In [33]:
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Dataset shape: (1124, 9)
Number of rows: 1124
Number of columns: 9


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nInfo:")
df.info()
print("\nData types:")
print(df.dtypes)

print("\nInfo:")
df.info()


Shape: (1124, 9)

Data types:
Area_SqFt        float64
Rooms            float64
Build_Year         int64
Location             str
Street_Type          str
Furnishing           str
Property_Type        str
Has_Pool             str
Price            float64
dtype: object

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Area_SqFt      1091 non-null   float64
 1   Rooms          1091 non-null   float64
 2   Build_Year     1124 non-null   int64  
 3   Location       1124 non-null   str    
 4   Street_Type    1124 non-null   str    
 5   Furnishing     1091 non-null   str    
 6   Property_Type  1124 non-null   str    
 7   Has_Pool       1124 non-null   str    
 8   Price          1124 non-null   float64
dtypes: float64(3), int64(1), str(5)
memory usage: 79.2 KB

Data types:
Area_SqFt        float64
Rooms            float64
Build_Year         int6

In [34]:
print("\n--- First 5 Rows ---")
print(df.head())


--- First 5 Rows ---
     Area_SqFt  Rooms  Build_Year Location       Street_Type      Furnishing  \
0  2473.192784    4.0        1992   Jaipur  Residential Lane       Furnished   
1  2353.472711    4.0        2006   Indore       Corner Plot     Unfurnished   
2  2212.222005    3.0        2012   Jaipur    Highway Facing  Semi-Furnished   
3  2823.886596    6.0        1993  Lucknow         Main Road     Unfurnished   
4  1869.648721    5.0        2012   Jaipur       Corner Plot  Semi-Furnished   

  Property_Type Has_Pool     Price  
0     Apartment       No  568486.0  
1     Apartment      Yes  577214.0  
2        Duplex       No  581300.0  
3         Villa      Yes  794614.0  
4     Apartment       No  493086.0  


**Data Cleaning and Preprocessing**

In [35]:
# Check for missing values in each column
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100

print("Missing values by column:")
print(missing_counts)

print("\nMissing values percentage by column:")
print(missing_percent.round(2))

Missing values by column:
Area_SqFt        33
Rooms            33
Build_Year        0
Location          0
Street_Type       0
Furnishing       33
Property_Type     0
Has_Pool          0
Price             0
dtype: int64

Missing values percentage by column:
Area_SqFt        2.94
Rooms            2.94
Build_Year       0.00
Location         0.00
Street_Type      0.00
Furnishing       2.94
Property_Type    0.00
Has_Pool         0.00
Price            0.00
dtype: float64


In [36]:
# Check the extent and distribution of missing data in key variables to understand 
# whether missing values occur individually or across multiple columns.
missing_mask = df[['Area_SqFt', 'Rooms', 'Furnishing']].isna()
print(missing_mask.sum())
print()
print("Rows missing all three:", (missing_mask.sum(axis=1) == 3).sum())
print("Rows missing exactly one:", (missing_mask.sum(axis=1) == 1).sum())
print("Rows missing exactly two:", (missing_mask.sum(axis=1) == 2).sum())

Area_SqFt     33
Rooms         33
Furnishing    33
dtype: int64

Rows missing all three: 0
Rows missing exactly one: 95
Rows missing exactly two: 2


In [37]:
# Detect outliers in all numeric columns using the Interquartile Range (IQR) method,
# then summarize the number of outliers per column and identify rows containing
# at least one outlier for further investigation.
numeric_cols = df.select_dtypes(include=[np.number]).columns

q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (df[numeric_cols] < lower_bound) | (df[numeric_cols] > upper_bound)
outlier_counts = outlier_mask.sum()

print("Outlier counts by numeric column:")
print(outlier_counts)

print("\nRows with any numeric outlier:")
outlier_rows = df[outlier_mask.any(axis=1)]
print(outlier_rows[numeric_cols].head(10))

print(f"\nTotal rows with at least one numeric outlier: {len(outlier_rows)}")

Outlier counts by numeric column:
Area_SqFt     18
Rooms          0
Build_Year     0
Price         15
dtype: int64

Rows with any numeric outlier:
       Area_SqFt  Rooms  Build_Year        Price
7    4316.818615    6.0        2008   988286.819
52   5648.321783    7.0        2003  1355674.054
134  3496.557001    7.0        2021  1031661.000
137   700.000000    6.0        1987   310136.000
211  5505.772234    5.0        1995  1284263.922
232  3223.723578    7.0        2008   981429.000
234   700.000000    5.0        2014   340017.000
286  6170.583006    2.0        2022  1265986.684
339  4122.732784    2.0        1987   783551.000
428  9022.275608    6.0        2001  1396460.448

Total rows with at least one numeric outlier: 23


In [ ]:
# Create a separate cleaned copy so the original dataset remains available for comparison
# This keeps all columns intact while improving data quality more carefully.
df_clean = df.copy()

# 1) Standardize missing values in text/object columns
# Some datasets store missing values as empty strings, 'None', or 'nan' text.
# Converting them to actual NaN values makes them easier to handle consistently.
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = df_clean[col].astype('string').str.strip()
    df_clean[col] = df_clean[col].replace({'nan': np.nan, 'None': np.nan, 'null': np.nan, '': np.nan})

# 2) Convert numeric-looking columns to numeric data types
# This helps avoid issues where numbers are stored as strings.
# The list below includes common house-price related numeric fields.
numeric_columns = [col for col in ['Area_SqFt', 'Rooms', 'Bedrooms', 'Bathrooms', 'Parking', 'Price'] if col in df_clean.columns]
for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 3) Standardize common placeholder text values in categorical columns
# This reduces the number of inconsistent category labels and makes later imputation more reliable.
for col in df_clean.select_dtypes(exclude=[np.number]).columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace({'unknown': np.nan, 'Unknown': np.nan, 'N/A': np.nan, 'NA': np.nan, 'none': np.nan, 'None': np.nan})

# 4) Handle impossible numeric values by treating them as missing
# Zero or negative values in house-related numeric features are often invalid or suspicious,
# so they are converted to NaN and then imputed later.
for col in ['Area_SqFt', 'Rooms', 'Bedrooms', 'Bathrooms', 'Parking', 'Price']:
    if col in df_clean.columns:
        df_clean.loc[df_clean[col] <= 0, col] = np.nan

# 5) Check for duplicate rows
# Duplicates can distort analysis and model training, so it is useful to flag them early.
duplicate_count = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# 6) Check for suspicious relationships between columns
# These are simple sanity checks to spot values that may be unrealistic.
if {'Bedrooms', 'Rooms'}.issubset(df_clean.columns):
    suspicious_bedroom_count = (df_clean['Bedrooms'] > df_clean['Rooms']).sum()
    print(f"Rows where Bedrooms > Rooms: {suspicious_bedroom_count}")

if {'Bathrooms', 'Rooms'}.issubset(df_clean.columns):
    suspicious_bathroom_count = (df_clean['Bathrooms'] > df_clean['Rooms']).sum()
    print(f"Rows where Bathrooms > Rooms: {suspicious_bathroom_count}")

# 7) Impute missing values using group-based statistics when possible
# This is often more accurate than using a global average because values can vary by category.
# It will first try to fill missing values using a category-based median or mode.
group_col = next((col for col in ['Furnishing', 'Location', 'City', 'Property_Type', 'Type', 'Neighborhood'] if col in df_clean.columns), None)

for col in df_clean.columns:
    if col == group_col:
        continue

    if pd.api.types.is_numeric_dtype(df_clean[col]):
        # Use the median within each group first, then fall back to the overall median.
        if group_col is not None:
            group_medians = df_clean.groupby(group_col)[col].transform('median')
            df_clean[col] = df_clean[col].fillna(group_medians)

        overall_median = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(overall_median)

    else:
        # Use the mode within each group first, then fall back to the overall mode.
        if group_col is not None:
            group_modes = df_clean.groupby(group_col)[col].transform(
                lambda s: s.mode(dropna=True).iloc[0] if not s.mode(dropna=True).empty else np.nan
            )
            df_clean[col] = df_clean[col].fillna(group_modes)

        mode_value = df_clean[col].mode(dropna=True)
        if not mode_value.empty:
            df_clean[col] = df_clean[col].fillna(mode_value.iloc[0])

# 8) Handle outliers more carefully by capping extreme values instead of dropping them
# This preserves the full dataset while reducing the effect of extreme values on analysis.
for col in df_clean.select_dtypes(include=[np.number]).columns:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1

    if pd.notna(iqr) and iqr != 0:
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

# 9) Review the cleaned result
# This gives you a quick check that missing values were handled properly.
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("\nRemaining missing values after cleaning:")
print(df_clean.isna().sum())


C:\Users\chris\AppData\Local\Temp\ipykernel_31300\3898836980.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_clean.select_dtypes(include=['object']).columns:


Original shape: (1124, 9)
Cleaned shape: (1124, 9)

Remaining missing values after cleaning:
Area_SqFt         0
Rooms             0
Build_Year        0
Location          0
Street_Type       0
Furnishing       33
Property_Type     0
Has_Pool          0
Price             0
dtype: int64


In [39]:
# Compare key numeric columns before and after cleaning
# This helps you check whether the cleaning process made the data more reasonable.

numeric_cols = [col for col in ['Area_SqFt', 'Rooms', 'Bedrooms', 'Bathrooms', 'Parking', 'Price'] if col in df.columns and col in df_clean.columns]

print("Summary of numeric columns before and after cleaning:")
print("-" * 70)

for col in numeric_cols:
    before = df[col]
    after = df_clean[col]
    print(f"{col}:")
    print(f"  Before -> mean: {before.mean():.2f}, median: {before.median():.2f}, std: {before.std():.2f}")
    print(f"  After  -> mean: {after.mean():.2f}, median: {after.median():.2f}, std: {after.std():.2f}")
    print()


Summary of numeric columns before and after cleaning:
----------------------------------------------------------------------
Area_SqFt:
  Before -> mean: 2237.48, median: 2201.55, std: 731.10
  After  -> mean: 2207.87, median: 2199.75, std: 557.42

Rooms:
  Before -> mean: 4.57, median: 5.00, std: 1.68
  After  -> mean: 4.57, median: 5.00, std: 1.66

Price:
  Before -> mean: 608201.51, median: 602541.00, std: 143195.81
  After  -> mean: 604688.34, median: 602541.00, std: 126637.91

